In [21]:
import torch
import torch.nn as nn
import numpy as np

def im2col_multi(X, kernel_shape, stride=1, padding=0):
    C, H, W = X.shape
    kH, kW = kernel_shape

    X_padded = np.pad(X, ( (0, 0),(padding, padding), (padding, padding) ), mode='constant')

    H_p, W_p = X_padded.shape[1:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    cols = []

    for i in range(0, out_H*stride, stride):
        for j in range(0, out_W * stride, stride):
            patch = X_padded[:, i:i+kH, j:j+kW].ravel()
            cols.append(patch)
    
    return np.array(cols), out_H, out_W

def col2im_multi(cols, output_shape, kernel_shape, stride=1, padding=0):
    C, H, W = output_shape
    kH, kW = kernel_shape
    H_p, W_p = H+2*padding, W+2*padding
    X_padded = np.zeros((C,H_p,W_p))

    out_H = (H_p - kH)//stride + 1
    out_W = (W_p - kW)//stride + 1

    idx = 0
    for i in range(0, out_H*stride, stride):
        for j in range(0, out_W*stride, stride):
            patch = cols[idx].reshape(C, kH, kW)
            X_padded[:, i:i+kH, j:j+kW] += patch
            idx += 1

    if padding>0:
        X_padded = X_padded[:, padding:-padding, padding:-padding]

    return X_padded

def conv2d_im2col_multi(X, W, stride=1, padding=0):
    
    C_out, C_in, kH, kW = W.shape
    X_col, out_H, out_W = im2col_multi(X, (kH, kW), stride, padding)

    W_col = W.reshape(C_out, -1)
    Y_col = X_col @ W_col.T

    Y = Y_col.T.reshape(C_out, out_H, out_W)
    return Y

def conv_transpose2d_img2col_multi(Y, W, stride=1, padding=0, output_shape=None):
    C_out, C_in, kH, kW = W.shape
    Y_col = Y.reshape(C_out, -1)
    W_col = W.reshape(C_out, -1)
    X_col = W_col.T @ Y_col

    if output_shape is None:
        H_out = (Y.shape[1]-1) * stride - 2*padding + kH
        W_out = (Y.shape[2]-1) * stride - 2*padding + kW
        output_shape = (C_in, H_out, W_out)

    X = col2im_multi(X_col.T, output_shape=output_shape, kernel_shape=(kH, kW), stride=stride, padding=padding)

    return X

In [22]:
torch.manual_seed(1)

C_out, C_in = 6, 3
input_size=15
kernel_size=3

stride=2
padding=3

x = torch.randn(C_in, input_size, input_size)
kernel = torch.ones(C_out, C_in, kernel_size, kernel_size)

print(f'x: {x.shape}')
print(f'Kernel: {kernel.shape}')

x: torch.Size([3, 15, 15])
Kernel: torch.Size([6, 3, 3, 3])


In [23]:
conv = nn.Conv2d(C_in, C_out, kernel_size=kernel_size, stride=stride, padding=padding)
convt = nn.ConvTranspose2d(6, 3, kernel_size=kernel_size, stride=stride, padding=padding)

with torch.no_grad():
    conv.weight.copy_(kernel)
    conv.bias.zero_()
    convt.weight.copy_(kernel)
    convt.bias.zero_()

output_conv = conv(x)
print(f'Conv out: {output_conv.shape}')

output_convt = convt(output_conv)
print(f'ConvT out: {output_convt.shape}')

Conv out: torch.Size([6, 10, 10])
ConvT out: torch.Size([3, 15, 15])


In [24]:
x = np.array(x)
kernel = np.array(kernel)

result = conv2d_im2col_multi(x, kernel, stride=stride, padding=padding)
print(f'result: {result.shape}')

original = conv_transpose2d_img2col_multi(result, kernel, stride=stride, padding=padding, output_shape=None)
print(f'original: {original.shape}')
#print(original)

result: (6, 10, 10)
original: (3, 15, 15)


C:\Users\glovric\AppData\Local\Temp\ipykernel_5532\1603341396.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  x = np.array(x)
C:\Users\glovric\AppData\Local\Temp\ipykernel_5532\1603341396.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  kernel = np.array(kernel)


In [25]:
print(output_conv[0][0])
print(result[0][0])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], grad_fn=<SelectBackward0>)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [26]:
print(output_convt[0][0])
print(original[0][0])

tensor([  2.5895,   5.8468,   3.2574,  21.5423,  18.2849,   2.7877, -15.4972,
        -18.0091,  -2.5119, -18.1458, -15.6339, -67.9971, -52.3633, -59.8906,
         -7.5274], grad_fn=<SelectBackward0>)
[  2.58945203   5.84680629   3.25735426  21.54226446  18.2849102
   2.78773785 -15.49717236 -18.00911379  -2.51194143 -18.14581919
 -15.63387775 -67.99713612 -52.36325836 -59.89062214  -7.52736378]
